# Video HLS Generator
Supports a **single video URL** or a **Google Drive folder** with multiple videos.

> **Tip:** Go to **Runtime → Change runtime type → T4 GPU** for full GPU pipeline.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install -q yt-dlp gdown

import subprocess

caps = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True)
USE_NVENC = "h264_nvenc" in caps.stdout

filters = subprocess.run(["ffmpeg", "-hide_banner", "-filters"], capture_output=True, text=True)
USE_NPP = "scale_npp" in filters.stdout

USE_GPU = USE_NVENC  # GPU encode path overall

if USE_NVENC and USE_NPP:
    print("✓ GPU detected — CUDA decode + scale_npp + NVENC encode (full GPU)")
elif USE_NVENC:
    print("✓ GPU detected — NVENC encode (CPU scaling, no scale_npp)")
else:
    print("⚠ No GPU found — using libx264 (CPU). Enable T4 in Runtime settings.")

In [ ]:
# ── Cloud Mode: load config from Drive (written by the desktop app) ──────────────
import json as _json

CLOUD_MODE = False
_cloud_config = {}

try:
    from google.colab import auth as _colab_auth
    _colab_auth.authenticate_user()
    from googleapiclient.discovery import build as _build
    _drive = _build('drive', 'v3', cache_discovery=False)

    _roots = _drive.files().list(
        q="name='VideoCompressor' and mimeType='application/vnd.google-apps.folder' and trashed=false",
        fields='files(id)'
    ).execute().get('files', [])

    if _roots:
        _root_id = _roots[0]['id']
        _cfgs = _drive.files().list(
            q=f"'{_root_id}' in parents and name='config' and mimeType='application/vnd.google-apps.folder'",
            fields='files(id)'
        ).execute().get('files', [])
        if _cfgs:
            _cfg_folder_id = _cfgs[0]['id']
            _jsons = _drive.files().list(
                q=f"'{_cfg_folder_id}' in parents and name='colab_config.json' and trashed=false",
                fields='files(id, modifiedTime)'
            ).execute().get('files', [])
            if _jsons:
                _raw = _drive.files().get_media(fileId=_jsons[0]['id']).execute()
                _cloud_config = _json.loads(_raw)
                CLOUD_MODE = True
                print(f"Cloud Mode ACTIVE")
                print(f"  Input:       {_cloud_config.get('input_filename')}")
                print(f"  Resolutions: {_cloud_config.get('selected_resolutions')}")
            else:
                print('No colab_config.json found — running in manual mode.')
        else:
            print('No config folder in VideoCompressor — running in manual mode.')
    else:
        print('No VideoCompressor folder in Drive — running in manual mode.')
except Exception as _e:
    print(f'Cloud config load skipped ({_e}) — running in manual mode.')


In [ ]:
# ── Configuration — edit these (or leave as-is when using Cloud Mode) ──────────────

if CLOUD_MODE:
    # Pre-filled by the desktop app via colab_config.json
    DRIVE_FOLDER_URL        = ''
    SINGLE_VIDEO_URL        = ''
    SELECTED_RESOLUTIONS    = _cloud_config.get('selected_resolutions', ['1080p', '720p'])
    _CLOUD_INPUT_FILE_ID    = _cloud_config.get('input_file_id')
    _CLOUD_INPUT_FILENAME   = _cloud_config.get('input_filename')
    _CLOUD_OUTPUT_FOLDER_ID = _cloud_config.get('output_folder_id')
    _CLOUD_CONFIG_FOLDER_ID = _cloud_config.get('config_folder_id')
    CPU_THREADS = 2
    print(f'[Cloud] Will download input by file ID: {_CLOUD_INPUT_FILE_ID}')
else:
    # Option A: Google Drive shared folder URL (processes ALL videos inside)
    DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1ojVwHDmIygJFNB4x74jPyS_W9VwTCEjq?usp=sharing'

    # Option B: Single direct video URL (YouTube, direct .mp4 link, etc.)
    # Set DRIVE_FOLDER_URL = '' and fill this instead
    SINGLE_VIDEO_URL = ''

    SELECTED_RESOLUTIONS = ['1080p', '720p', '480p', '360p']  # remove any you don't need
    CPU_THREADS = 2  # only used when GPU is not available


In [ ]:
# ── Status reporter (Cloud Mode only) ───────────────────────────────────────────────────────────
from googleapiclient.http import MediaIoBaseUpload
import datetime as _dt, io as _io

def _write_status(state: str, message: str, progress_pct: int = 0):
    """Write colab_status.json to Drive so the desktop app can poll progress."""
    if not CLOUD_MODE:
        return
    try:
        data = {
            'state': state,
            'message': message,
            'progress_pct': progress_pct,
            'updated_at': _dt.datetime.now(_dt.timezone.utc).isoformat(),
        }
        content = _json.dumps(data).encode()
        media = MediaIoBaseUpload(_io.BytesIO(content), mimetype='application/json')
        # Delete old status file first
        old = _drive.files().list(
            q=f"'{_CLOUD_CONFIG_FOLDER_ID}' in parents and name='colab_status.json' and trashed=false",
            fields='files(id)'
        ).execute().get('files', [])
        for _f in old:
            _drive.files().delete(fileId=_f['id']).execute()
        _drive.files().create(
            body={'name': 'colab_status.json', 'parents': [_CLOUD_CONFIG_FOLDER_ID]},
            media_body=media, fields='id'
        ).execute()
    except Exception as _e:
        print(f'[Status] Could not write status: {_e}')

print('Status reporter ready.')


In [ ]:
# ── List Drive folder & select files to download ──────────────────────────────
import re, os, shutil, gdown
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

VIDEO_EXTENSIONS = {'.mp4', '.mov', '.mkv', '.avi', '.webm', '.flv', '.m4v'}
VIDEO_MIME_TYPES  = {'video/mp4', 'video/quicktime', 'video/x-msvideo',
                     'video/x-matroska', 'video/webm', 'video/x-flv', 'video/x-m4v'}

file_checkboxes = {}   # {file_id: {cb, name, id}}

if CLOUD_MODE:
    # Desktop app already placed the file in Drive; we'll download it directly in the next cell.
    print(f'[Cloud Mode] Input: {_CLOUD_INPUT_FILENAME} — will download by file ID.')
    print('Skip to the Download cell below (it runs automatically).')

elif DRIVE_FOLDER_URL:
    # Authenticate with Google (one-click in Colab)
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build

    drive_service = build('drive', 'v3', cache_discovery=False)

    # Extract folder ID from URL
    match = re.search(r'/folders/([a-zA-Z0-9_-]+)', DRIVE_FOLDER_URL)
    if not match:
        raise ValueError('Could not extract folder ID from DRIVE_FOLDER_URL')
    folder_id = match.group(1)

    # List all video files in the folder
    results = drive_service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields='files(id, name, mimeType, size)',
        pageSize=100
    ).execute()

    all_files = results.get('files', [])
    video_files_meta = [
        f for f in all_files
        if f.get('mimeType', '') in VIDEO_MIME_TYPES
        or Path(f.get('name', '')).suffix.lower() in VIDEO_EXTENSIONS
    ]
    video_files_meta.sort(key=lambda f: f['name'])

    if not video_files_meta:
        raise RuntimeError('No video files found in the Drive folder.')

    # Build checkboxes
    select_all_btn = widgets.ToggleButton(
        value=True, description='Select All', button_style='info',
        layout=widgets.Layout(width='120px', margin='0 0 10px 0')
    )
    for meta in video_files_meta:
        size_mb = int(meta.get('size', 0)) / 1024 / 1024
        label = f"{meta['name']}  ({size_mb:.1f} MB)" if size_mb > 0 else meta['name']
        cb = widgets.Checkbox(value=True, description=label,
                              layout=widgets.Layout(width='550px'))
        file_checkboxes[meta['id']] = {'cb': cb, 'name': meta['name'], 'id': meta['id']}

    def toggle_all(change):
        for v in file_checkboxes.values():
            v['cb'].value = change['new']
    select_all_btn.observe(toggle_all, names='value')

    print(f'Found {len(video_files_meta)} video(s) in Drive folder.')
    print('Check the files you want to download, then run the next cell.\n')
    display(select_all_btn)
    display(widgets.VBox([v['cb'] for v in file_checkboxes.values()]))

else:
    print('Single URL mode — skip to the next cell.')


In [ ]:
# ── Download selected files ───────────────────────────────────────────────────
os.makedirs('input', exist_ok=True)
video_files = []

if CLOUD_MODE:
    _write_status('running', 'Downloading input video...', 5)
    out_path = os.path.join('input', _CLOUD_INPUT_FILENAME)
    print(f'[Cloud] Downloading input: {_CLOUD_INPUT_FILENAME}')
    gdown.download(id=_CLOUD_INPUT_FILE_ID, output=out_path, quiet=False)
    video_files = [out_path]
    print(f'[Cloud] Input ready: {out_path}')

elif DRIVE_FOLDER_URL:
    selected = [v for v in file_checkboxes.values() if v['cb'].value]
    if not selected:
        raise ValueError('No files selected. Go back and check at least one box.')

    print(f'Downloading {len(selected)} file(s)...\n')
    for item in selected:
        out_path = os.path.join('input', item['name'])
        print(f'  \u2193 {item["name"]}')
        gdown.download(id=item['id'], output=out_path, quiet=True)
        video_files.append(out_path)

elif SINGLE_VIDEO_URL:
    print(f'Downloading: {SINGLE_VIDEO_URL}')
    result = subprocess.run(
        ['yt-dlp', '--no-playlist', '-o', 'input/%(title)s.%(ext)s', SINGLE_VIDEO_URL],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        filename = SINGLE_VIDEO_URL.split('/')[-1].split('?')[0] or 'video.mp4'
        subprocess.run(['wget', '-q', '-O', f'input/{filename}', SINGLE_VIDEO_URL])
    video_files = sorted([
        str(p) for p in Path('input').iterdir()
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    ])
else:
    raise ValueError('Set either DRIVE_FOLDER_URL or SINGLE_VIDEO_URL in the config cell.')

print(f'\n\u2713 Ready to process {len(video_files)} file(s):')
for v in video_files:
    print(f'  \u2022 {Path(v).name}')


In [ ]:
# ── HLS Generation ────────────────────────────────────────────────────────────

RESOLUTIONS = {
    '1080p': {'width': 1920, 'height': 1080, 'bitrate': '5000k', 'audio': '192k'},
    '720p':  {'width': 1280, 'height': 720,  'bitrate': '2800k', 'audio': '128k'},
    '480p':  {'width': 854,  'height': 480,  'bitrate': '1400k', 'audio': '128k'},
    '360p':  {'width': 640,  'height': 360,  'bitrate': '800k',  'audio': '96k'},
}

MASTER_BANDWIDTHS = {
    '1080p': 5000000, '720p': 2800000, '480p': 1400000, '360p': 800000,
}

def _double_bitrate(bitrate: str) -> str:
    return str(int(bitrate.rstrip('k')) * 2) + 'k'

def _cap_bitrate(bitrate_str: str, source_kbps: int) -> str:
    target_k = int(bitrate_str.rstrip('k'))
    return f'{min(target_k, max(source_kbps, 200))}k'

def _probe_video(video_path):
    r = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
         '-show_entries', 'stream=width,height,bit_rate',
         '-of', 'default=noprint_wrappers=1', video_path],
        capture_output=True, text=True
    )
    info = dict(line.split('=') for line in r.stdout.strip().splitlines() if '=' in line)
    w = int(info.get('width', 1920))
    h = int(info.get('height', 1080))
    kbps = int(info.get('bit_rate', 0) or 0) // 1000
    return w, h, kbps

print(f'Processing {len(video_files)} file(s):\n')

def encode_video(video_path, selected_res, video_index=0, total_videos=1):
    stem = Path(video_path).stem
    job_dir = os.path.join('output', stem)
    os.makedirs(job_dir, exist_ok=True)

    probe = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', video_path],
        capture_output=True, text=True
    )
    try:
        duration = float(probe.stdout.strip())
        print(f'  Duration: {duration:.1f}s')
    except Exception:
        duration = 0.0

    src_w, src_h, src_kbps = _probe_video(video_path)
    print(f'  Source: {src_w}x{src_h} @ {src_kbps}kbps')

    base_pct = int(video_index / total_videos * 80) + 10  # range: 10–90
    _write_status('running', f'Encoding {stem}...', base_pct)

    # Skip resolutions taller than the source to avoid useless upscaling
    active_res = [r for r in selected_res if RESOLUTIONS[r]['height'] <= src_h]
    if not active_res:
        active_res = [selected_res[-1]]
    skipped = set(selected_res) - set(active_res)
    if skipped:
        print(f'  Skipping {', '.join(skipped)} (above source resolution)')

    thumb_path = os.path.join(job_dir, 'thumbnail.jpg')
    subprocess.run(
        ['ffmpeg', '-y', '-ss', '1', '-i', video_path,
         '-vframes', '1', '-q:v', '2', thumb_path],
        capture_output=True
    )

    for res in active_res:
        os.makedirs(os.path.join(job_dir, res), exist_ok=True)

    n = len(active_res)

    if USE_NPP:
        scale_filter = 'scale_npp'
        scale_suffix = ''
    else:
        scale_filter = 'scale'
        scale_suffix = ':flags=fast_bilinear'

    split_labels = ''.join(f'[v{i}]' for i in range(n))
    filter_parts = [f'[0:v]split={n}{split_labels}']
    for i, res in enumerate(active_res):
        cfg = RESOLUTIONS[res]
        filter_parts.append(
            f'[v{i}]{scale_filter}={cfg["width"]}:{cfg["height"]}{scale_suffix}[s{i}]'
        )
    filter_complex = ';'.join(filter_parts)

    if USE_NPP:
        hwaccel_flags = ['-hwaccel', 'cuda', '-hwaccel_output_format', 'cuda']
    elif USE_NVENC:
        hwaccel_flags = ['-hwaccel', 'cuda']
    else:
        hwaccel_flags = []

    base_cmd = ['ffmpeg', '-y', *hwaccel_flags, '-i', video_path,
                '-filter_complex', filter_complex]
    if not USE_GPU:
        base_cmd += ['-threads', str(CPU_THREADS)]

    per_output = []
    for i, res in enumerate(active_res):
        cfg = RESOLUTIONS[res]
        segment_pat = os.path.join(job_dir, res, '%03d.ts')
        playlist    = os.path.join(job_dir, res, 'playlist.m3u8')

        video_br = _cap_bitrate(cfg['bitrate'], src_kbps) if src_kbps > 0 else cfg['bitrate']

        if USE_NVENC:
            video_flags = ['-c:v', 'h264_nvenc', '-preset', 'p1',
                           '-rc', 'cbr', '-b:v', video_br, '-maxrate', video_br,
                           '-b_ref_mode', 'disabled',
                           '-profile:v', 'main']
        else:
            video_flags = ['-c:v', 'libx264', '-crf', '23', '-preset', 'veryfast',
                           '-profile:v', 'main',
                           '-maxrate', video_br, '-bufsize', _double_bitrate(video_br)]

        per_output += [
            '-map', f'[s{i}]', '-map', '0:a?',
            *video_flags,
            '-c:a', 'aac', '-b:a', cfg['audio'],
            '-hls_time', '10', '-hls_playlist_type', 'vod',
            '-hls_segment_filename', segment_pat,
            playlist,
        ]

    cmd = base_cmd + per_output

    res_label = '+'.join(active_res)
    if USE_NVENC and USE_NPP:
        encoder_label = 'NVENC p1 (full GPU)'
    elif USE_NVENC:
        encoder_label = 'NVENC p1 + fast_bilinear scale'
    else:
        encoder_label = f'libx264 veryfast ({CPU_THREADS}t)'
    print(f'  Encoding [{res_label}] via {encoder_label} in one pass...', end='', flush=True)

    process = subprocess.Popen(cmd, stderr=subprocess.PIPE, stdout=subprocess.DEVNULL,
                               text=True, bufsize=1)

    stderr_lines = []
    last_pct = -1
    for line in process.stderr:
        stderr_lines.append(line)
        m = re.search(r'time=(\d+):(\d+):([\d.]+)', line)
        if m and duration:
            elapsed = int(m.group(1))*3600 + int(m.group(2))*60 + float(m.group(3))
            pct = min(int(elapsed / duration * 100), 100)
            if pct != last_pct and pct % 10 == 0:
                print(f' {pct}%', end='', flush=True)
                overall = base_pct + int(pct * 0.8 / total_videos)
                _write_status('running', f'Encoding {stem}: {pct}%', overall)
                last_pct = pct

    process.wait()
    if process.returncode != 0:
        print(f'\n[FFmpeg failed — exit {process.returncode}]')
        print(''.join(stderr_lines[-40:]))
        raise RuntimeError(f'FFmpeg encoding failed (exit {process.returncode})')
    print(' ✓')

    master_path = os.path.join(job_dir, 'master.m3u8')
    with open(master_path, 'w') as f:
        f.write('#EXTM3U\n#EXT-X-VERSION:3\n')
        for res in active_res:
            cfg = RESOLUTIONS[res]
            f.write(f'#EXT-X-STREAM-INF:BANDWIDTH={MASTER_BANDWIDTHS[res]},'
                    f'RESOLUTION={cfg["width"]}x{cfg["height"]}\n')
            f.write(f'{res}/playlist.m3u8\n')

    print(f'  ✓ master.m3u8 written')
    return job_dir


os.makedirs('output', exist_ok=True)
completed = []

for idx, video_path in enumerate(video_files):
    stem = Path(video_path).stem
    print(f'[{idx+1}/{len(video_files)}] Processing: {stem}')
    job_dir = encode_video(video_path, SELECTED_RESOLUTIONS, idx, len(video_files))
    completed.append((stem, job_dir))
    print(f'  ✓ Done → {job_dir}\n')

print(f'=== All {len(completed)} video(s) processed ===')


In [ ]:
# ── Zip and upload/download outputs ─────────────────────────────────────────────────
_write_status('running', 'Zipping output...', 92)

if len(completed) == 1:
    stem, job_dir = completed[0]
    zip_path = shutil.make_archive(f'{stem}_hls', 'zip', 'output', stem)
else:
    zip_path = shutil.make_archive('hls_output_all', 'zip', '.', 'output')

zip_name = os.path.basename(zip_path)
zip_size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f'Zipped → {zip_path} ({zip_size_mb:.1f} MB)')

if CLOUD_MODE:
    from googleapiclient.http import MediaFileUpload as _MediaFileUpload
    _write_status('running', 'Uploading zip to Drive...', 95)
    print(f'[Cloud] Uploading {zip_name} to Drive output folder...')
    _media = _MediaFileUpload(zip_path, mimetype='application/zip', resumable=True)
    _drive.files().create(
        body={'name': zip_name, 'parents': [_CLOUD_OUTPUT_FOLDER_ID]},
        media_body=_media, fields='id'
    ).execute()
    _write_status('complete', f'Done. Output: {zip_name}', 100)
    print('[Cloud] Upload complete. The desktop app will auto-download the result.')
else:
    from google.colab import files
    files.download(zip_path)


In [ ]:
# ── Cleanup: delete input folder ─────────────────────────────────────────────
import shutil, os

if os.path.exists("input"):
    shutil.rmtree("input")
    print("Input folder deleted.")
else:
    print("Input folder does not exist.")

In [ ]:
# ── Cleanup: delete output folder ────────────────────────────────────────────
import shutil, os

if os.path.exists("output"):
    shutil.rmtree("output")
    print("Output folder deleted.")
else:
    print("Output folder does not exist.")